# 1 LangGraph 总览

本教程回答三个问题：

1. LangChain 和 LangGraph 是什么关系？该用哪个？（本章）
2. 一个 Graph 由哪些基本要素构成？（第 2 章）
3. Graph 是怎么运行起来的？（第 3 章）

## 1.1 LangChain 与 LangGraph

### 1.1.1 它们是什么

- **LangChain**：构建 LLM 应用的组件库（model 调用、prompt template、output parser、document 处理、vector store 等）
- **LangGraph**：LangChain 生态里的 Graph 编排框架，以「Graph + State」为核心，解决了早期 Chain 的局限

### 1.1.2 为什么需要 LangGraph

LangChain 早期的 **Chain** 是线性管道 A → B → C，真实场景很快不够用：

- 没有**循环**：ReAct 这类 agent 需要「思考 → 行动 → 观察 → 再思考」
- 没有**分支**：不同输入走不同路径
- 没有**State**：步骤之间无法保存和更新上下文，更无法持久化
- 没有**人工介入**：无法在关键 Node 停下来等用户确认

LangGraph 用 Graph 取代 Chain：Node 是步骤，Edge 是路径，State 在整个 Graph 上流动，循环、分支、人工介入都直接支持。

### 1.1.3 定位：不是替代，是分工

| 维度 | LangChain | LangGraph |
|---|---|---|
| 定位 | 通用组件库 | 流程编排框架 |
| 核心抽象 | Chain / Runnable | StateGraph（directed graph） |
| 执行模型 | 线性管道 | Graph 遍历，支持循环/分支/并发 |
| State 管理 | 无（数据在 Chain 内传递） | 显式 State，可增量更新、可持久化 |
| 典型场景 | 简单问答、RAG pipeline | Agent、多 agent、人工审批流程 |
| 关系 | 提供组件 | 使用 LangChain 组件编排 |

**结论**：

- 固定线性步骤 → LangChain Chain 就够
- 需要循环、分支、State、人工介入 → LangGraph
- 复杂 agent 的主流选择是 LangGraph

# 2 Graph 的基本要素

一个 Graph 由 4 个要素构成：

| 要素 | 含义 | 代码对应 |
|---|---|---|
| **State** | 所有 Node 共享的数据容器 | 一个 TypedDict 或 Pydantic 类 |
| **Node** | 一个工作单元：普通 Python 函数 | `graph.add_node(...)` |
| **Edge** | Node 间的连接路径，决定执行顺序 | `graph.add_edge(...)` / `add_conditional_edges(...)` |
| **START / END** | 特殊 Node：Graph 的入口和出口 | `START`、`END` 常量 |

## 2.1 State

State 是一个类型定义，描述 Graph 上流动的数据。每个 Node 读 State、返回**部分更新**，LangGraph 自动把更新合并回 State：

```python
from typing_extensions import TypedDict

class ChatState(TypedDict):
    messages: list[str]   # 对话历史
    count: int            # 处理轮数
```

## 2.2 Node

Node 就是一个函数：**接收 State，返回要更新的字段**：

```python
def node_a(state: ChatState) -> dict:
    return {"count": state["count"] + 1}   # 只返回要更新的字段
```

## 2.3 Edge

- **普通 Edge**：`A → B`，A 跑完一定跑 B
- **条件 Edge**：A 跑完后根据返回值**选择**下一个 Node

## 2.4 拼装 StateGraph

```python
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)          # 1. 创建 Graph，声明 State 类型
builder.add_node("node_a", node_a)      # 2. 添加 Node
builder.add_edge(START, "node_a")       # 3. 连接：入口 → Node
builder.add_edge("node_a", END)         # 3. 连接：Node → 出口
app = builder.compile()                  # 4. 编译
```

下面用完整代码演示一个「两个 Node + State 累积」的最小 Graph：

# 3 Graph 运行过程

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str


def node_1(state: OverAllState) -> dict:
    return {
        "cur_id": state["cur_id"] + ", node_1",
        "logs": ["node_1 运行完毕"],
    }


def node_2(state: OverAllState) -> dict:
    return {
        "cur_id": state["cur_id"] + ", node_2",
        "logs": ["node_2 运行完毕"],
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

result = builder.compile().invoke({"cur_id": "start", "logs": []})
print(result)
# {'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}

# 4 State 管理

这一章深入 State：多个 Node 写同一个 key 时怎么合并，由 reducer 决定。

## 4.1 State Reducer

### 4.1.1 什么是 Reducer

Graph 运行时把 State 按 **key（channel）** 逐个合并。Node 返回的 dict 就是「State 更新」，同一个 key 被多个 Node 写入时，怎么合并由该 key 的 **reducer** 决定：

- **不加 reducer**：后写直接覆盖先写（last-writer-wins）
- **加 reducer**：`新值 = reducer(当前值, 更新值)`，Node 可以只返回增量

reducer 就是一个普通函数：`(当前值, 更新值) -> 新值`，通过 `Annotated[T, reducer]` 挂到字段上。

下面用同一个 Graph、两种 schema 对比：

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 两个 Node 连续写同一个 key
def node_inc(state) -> dict:
    return {"count": state["count"] + 1}


def node_add10(state) -> dict:
    return {"count": 10}


# 1) 无 reducer：后写覆盖
class OverwriteState(TypedDict):
    count: int


builder = StateGraph(state_schema=OverwriteState)
builder.add_node("inc", node_inc)
builder.add_node("add10", node_add10)
builder.add_edge(START, "inc")
builder.add_edge("inc", "add10")
builder.add_edge("add10", END)
result = builder.compile().invoke({"count": 0})
print("无 reducer（后写覆盖）:", result["count"])


# 2) 带 reducer：累加合并
class AddState(TypedDict):
    count: Annotated[int, add]


builder = StateGraph(state_schema=AddState)
builder.add_node("inc", node_inc)
builder.add_node("add10", node_add10)
builder.add_edge(START, "inc")
builder.add_edge("inc", "add10")
builder.add_edge("add10", END)
result = builder.compile().invoke({"count": 0})
print("带 reducer（累加合并）:", result["count"])
# 无 reducer（后写覆盖）: 10
# 带 reducer（累加合并）: 11

### 4.1.2 自定义 Reducer

自定义 reducer 只需写一个 `(current, update) -> new` 的纯函数，再用 `Annotated[T, fn]` 挂到字段上：

- 第一个参数是旧值，第二个参数是更新值
- **必须是纯函数**：不要做 IO、不要依赖外部可变状态（重放和并行场景会多次调用）
- 并行 Node 同时写同一个带 reducer 的 key 时，更新按任务顺序链式合并，结果要与顺序无关

下面演示两个自定义 reducer：dict 合并、取最大值：

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 自定义 reducer：dict 合并 / 取最大值
def merge_dict(a: dict, b: dict) -> dict:
    return {**a, **b}


def take_max(a: float, b: float) -> float:
    return max(a, b)


class ScoreState(TypedDict):
    cfg: Annotated[dict, merge_dict]
    score: Annotated[float, take_max]


# 两个并行 Node 各自写一部分
def node_a(state: ScoreState) -> dict:
    return {"cfg": {"model": "gpt-4"}, "score": 0.7}


def node_b(state: ScoreState) -> dict:
    return {"cfg": {"temperature": 0.2}, "score": 0.9}


builder = StateGraph(state_schema=ScoreState)
builder.add_node("a", node_a)
builder.add_node("b", node_b)
builder.add_edge(START, "a")
builder.add_edge(START, "b")
builder.add_edge("a", END)
builder.add_edge("b", END)

print(builder.compile().invoke({"cfg": {}, "score": 0}))
# {'cfg': {'model': 'gpt-4', 'temperature': 0.2}, 'score': 0.9}

### 4.1.3 内置 Reducer

LangGraph 内置了几个常用 reducer：

| Reducer | 适用字段 | 行为 |
|---|---|---|
| `operator.add` | `list` / `int` | list **拼接**；int **相加**（计数器） |
| `add_messages` | message 列表 | 按 `message.id` 去重：同 id **替换**，新 id **追加**；`RemoveMessage(id=...)` 删除 |
| `Overwrite(value)` | 任意带 reducer 的字段 | 绕过 reducer **直接覆盖**；同一 super-step 内多个 `Overwrite` 会报错 |

`add_messages`（`from langgraph.graph.message import add_messages`）是聊天场景的核心，LangGraph 还提供了现成的 `MessagesState`；`Overwrite` 从 `langgraph.types` 导入。

下面逐个看它们的运作逻辑。

#### 4.1.3.1 `operator.add`：list 拼接 / int 相加

`operator.add` 是最简单的 reducer：**新值 = 旧值 + 更新值**。list 是**拼接**，int 是**相加**（适合做计数器）。

先看 list 拼接：两个 Node 先后往同一个 key 追加日志。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    logs: Annotated[list[str], add]


def log_a(state: LogState) -> dict:
    return {"logs": ["node_a 运行"]}


def log_b(state: LogState) -> dict:
    return {"logs": ["node_b 运行"]}


builder = StateGraph(state_schema=LogState)
builder.add_node("a", log_a)
builder.add_node("b", log_b)
builder.add_edge(START, "a")
builder.add_edge("a", "b")
builder.add_edge("b", END)

print(builder.compile().invoke({"logs": []}))
# {'logs': ['node_a 运行', 'node_b 运行']}

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 同样的 add 挂到 int 字段上，就是计数器
class CountState(TypedDict):
    count: Annotated[int, add]


def inc_1(state: CountState) -> dict:
    return {"count": 1}


def inc_10(state: CountState) -> dict:
    return {"count": 10}


builder = StateGraph(state_schema=CountState)
builder.add_node("inc_1", inc_1)
builder.add_node("inc_10", inc_10)
builder.add_edge(START, "inc_1")
builder.add_edge("inc_1", "inc_10")
builder.add_edge("inc_10", END)

print(builder.compile().invoke({"count": 0}))
# {'count': 11}

#### 4.1.3.2 `add_messages`：按 id 去重合并

聊天场景专用 reducer，按 `message.id` 判断：

- **同 id** → 新消息**替换**旧消息
- **新 id** → **追加**到尾部
- `RemoveMessage(id=...)` → 按 id **删除**

In [ ]:
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, RemoveMessage
from langgraph.graph.message import add_messages

base: list[AnyMessage] = [HumanMessage(content="你好", id="1")]
update: list[AnyMessage] = [
    HumanMessage(content="你好呀", id="1"),            # 同 id → 替换
    AIMessage(content="有什么可以帮你？", id="2"),     # 新 id → 追加
]
# add_messages 签名类型较宽，这里显式收窄为 list[AnyMessage]；type: ignore 抑制签名检查
merged: list[AnyMessage] = add_messages(base, update)  # type: ignore
print("合并结果:", [m.content for m in merged])
# 合并结果: ['你好呀', '有什么可以帮你？']

# 按 id 删除
after_delete: list[AnyMessage] = add_messages(merged, [RemoveMessage(id="1")])  # type: ignore
print("删除后:", [m.content for m in after_delete])
# 删除后: ['有什么可以帮你？']

#### 4.1.3.3 `Overwrite`：绕过 reducer 直接覆盖

想**整体重置**某个字段时（比如一键清空对话），把更新值用 `Overwrite(value)` 包一层，LangGraph 会跳过 reducer，直接把字段写成 `value`。

注意：同一 super-step 里多个 Node 对同一个 key 写 `Overwrite` 会直接报错——整体覆盖无法合并，只能有一个写入者。

下面用现成的 `MessagesState` 演示清空对话：

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Overwrite


# MessagesState = {"messages": Annotated[list[AnyMessage], add_messages]}
def reply(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="我是助手")]}


def clear(state: MessagesState) -> dict:
    # 直接返回会走 add_messages 追加；Overwrite 包一层 → 整体替换
    return {"messages": Overwrite([SystemMessage(content="对话已清空")])}


builder = StateGraph(state_schema=MessagesState)
builder.add_node("reply", reply)
builder.add_node("clear", clear)
builder.add_edge(START, "reply")
builder.add_edge("reply", "clear")
builder.add_edge("clear", END)

result = builder.compile().invoke({"messages": [HumanMessage(content="你好")]})
print("清空后:", [m.content for m in result["messages"]])
# 清空后: ['对话已清空']

## 4.2 Node 中访问 State

Node 函数接收 State、返回更新。这一节看 Node 里怎么读 State、怎么写 State。

### 4.2.1 读取 State

Node 的第一个参数就是 State 本身，按 key 读取。State 是快照：Node 内部就地改它**不会生效**，想改只能靠返回值（见 4.2.2）。

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class ReadState(TypedDict):
    user: str
    count: int


def greet(state: ReadState) -> dict:
    print("读取 user:", state["user"])        # 按 key 读
    print("读取 count:", state.get("count"))  # 用 get 读
    return {}                                 # 什么都不改


builder = StateGraph(state_schema=ReadState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

builder.compile().invoke({"user": "Alice", "count": 1})
# 读取 user: Alice
# 读取 count: 1

### 4.2.2 更新 State

Node 通过**返回值**更新 State：dict 里只写要更新的 key，其余 key 保持不变；带 reducer 的 key 走 reducer 合并。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class UpdateState(TypedDict):
    user: str
    logs: Annotated[list[str], add]


def run(state: UpdateState) -> dict:
    return {"logs": ["step 完成"]}  # 只返回要更新的 key


builder = StateGraph(state_schema=UpdateState)
builder.add_node("run", run)
builder.add_edge(START, "run")
builder.add_edge("run", END)

print(builder.compile().invoke({"user": "Alice", "logs": []}))
# {'user': 'Alice', 'logs': ['step 完成']}

### 4.2.3 Overwrite 绕过 reducer

带 reducer 的 key 默认走合并；想**整体覆盖**时，把返回值用 `Overwrite(value)` 包一层（清空对话的示例见 4.1.3.3）。这里用计数器演示：直接返回会累加，`Overwrite` 则整体重置。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Overwrite


class CounterState(TypedDict):
    count: Annotated[int, add]


def inc(state: CounterState) -> dict:
    return {"count": 1}


def reset(state: CounterState) -> dict:
    return {"count": Overwrite(0)}  # 绕过 add，整体重置为 0


builder = StateGraph(state_schema=CounterState)
builder.add_node("inc", inc)
builder.add_node("reset", reset)
builder.add_edge(START, "inc")
builder.add_edge("inc", "reset")
builder.add_edge("reset", END)

print(builder.compile().invoke({"count": 5}))
# {'count': 0}   ← 不带 Overwrite 的话会是 6

# 5 Multi Schema

一个 Graph 可以有多个 schema：全局 State、输入 State、输出 State，以及每个 Node 的私有 State，分别约束 State 的共享范围、invoke 入参、返回值和 Node 可见性。

## 5.1 State 的类型和关系

| 类型 | 声明位置 | 作用 |
|---|---|---|
| 全局 State | `StateGraph(state_schema=...)` | 所有 Node 共享，贯穿整个 run |
| 输入 State | `StateGraph(input_schema=...)` | 约束 `invoke()` 入参 |
| 输出 State | `StateGraph(output_schema=...)` | 约束 `invoke()` 返回值 |
| 私有 State | `add_node(..., input_schema=...)` | 仅该 Node 可见 |

关系：

- 输入 State 与全局 State **同名**的 key 会写入全局，其余 key 只做入参校验、不进入全局
- 输出 State 的 key 必须存在于全局 State，否则 `invoke()` 返回 `None`
- 私有 schema 声明了全局同名 key 时，该 key 对 Node 可见；只属于私有 schema 的 key，其它 Node 看不到，也不进入输出
- 私有 State 只在单次 run 内有效：同一 run 中 Node 被多次执行时累积，跨 invoke 不保留

## 5.2 State 设计规范

- 全局 State 只放需要跨 Node 共享、或最终需要返回的数据
- 输入 State 只暴露调用方必须提供的字段；内部生成的字段放全局 State，由 Graph 自己初始化
- 输出 State 只暴露调用方需要的结果，key 必须在全局 State 中
- 私有 State 放单个 Node 的中间数据（循环累积、重试计数等），不跨 Node 共享的数据不要放全局
- Node 要读共享 key 时，把该 key 声明进私有 schema，否则 Node 看不到
- Node 返回值中：key 在全局 State 里 → 合并进全局；否则 → 进该 Node 的私有 channel

## 5.3 案例：全局 / 输入 / 输出 / 私有 State

一个问答 Graph：invoke 只接收 question，只返回 answer；plan Node 用私有 State 记录自己的循环计数，同时把全局 step 累计到 3 轮后转给 answer Node。

In [11]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


# 全局 State：Node 之间共享
class OverallState(TypedDict):
    question: str
    step: Annotated[int, add]
    answer: str


# 输入 State：invoke 只接收 question
class InputState(TypedDict):
    question: str


# 输出 State：invoke 只返回 answer
class OutputState(TypedDict):
    answer: str


# 私有 State：step 与全局同名（共享），tries 仅 plan 可见
class PlanState(TypedDict):
    step: Annotated[int, add]
    tries: Annotated[int, add]


def plan(state: PlanState) -> dict:
    print(f"第 {state['step']} 轮思考，尝试次数 {state['tries']}")
    return {"tries": 1, "step": 1}


def answer(state: OverallState) -> dict:
    return {"answer": f"{state['question']} -> 思考 {state['step']} 轮"}


def route(state: OverallState) -> str:
    return "plan" if state["step"] < 3 else "answer"


builder = StateGraph(
    state_schema=OverallState,
    input_schema=InputState,
    output_schema=OutputState,
)
builder.add_node("plan", plan, input_schema=PlanState)
builder.add_node("answer", answer)
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", route, {"plan": "plan", "answer": "answer"})
builder.add_edge("answer", END)

result = builder.compile().invoke({"question": "Q1"})
print(result)
# {'answer': 'Q1 -> 思考 3 轮'}

第 0 轮思考，尝试次数 0
第 1 轮思考，尝试次数 1
第 2 轮思考，尝试次数 2
{'answer': 'Q1 -> 思考 3 轮'}


# 6 预定义状态

LangGraph 提供了一些预定义 State，聊天场景最常用的是 `MessagesState`：`{"messages": Annotated[list[AnyMessage], add_messages]}`。继承它并加字段，就得到自定义的聊天 State。

In [20]:
from typing import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv(verbose=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


# 输入 State：invoke 只接收 username
class ChatInputState(TypedDict):
    username: str


# 全局 State：继承 MessagesState（自带 messages），再加两个字段
class ChatState(MessagesState):
    username: str
    output: str


# 输入 State：invoke 只接收 username
class ChatOutputState(TypedDict):
    response: str


def init_node(state: ChatInputState) -> dict:
    return {"messages": [HumanMessage("你好,我是" + state["username"])]}


def llm_node(state: ChatState) -> dict:
    response = model.invoke(state["messages"])
    return {
        "messages": [response],
        "response": response.content if response.content else "",
    }


builder = StateGraph(
    state_schema=ChatState, input_schema=ChatInputState, output_schema=ChatOutputState
)
builder.add_node("init", init_node)
builder.add_node("llm", llm_node)
builder.add_edge(START, "init")
builder.add_edge("init", "llm")
builder.add_edge("llm", END)

graph = builder.compile()

result = graph.invoke({"username": "小明"})
rprint(result)

{
    'response': 
'你好呀，小明！很高兴认识你。😊\n\n我是DeepSeek，你的AI助手。无论你是想聊天、学习新知识、解决工作学习中的问题，还是
需要创作灵感，我都在这里随时待命！\n\n今天有什么我可以帮你的吗？或者，想随便聊点什么也完全OK～'
}